In [ ]:
# =========================================
# CELL 1: Import required libraries
# =========================================
# Purpose:
# - Load all dependencies for image processing and display

import cv2
import numpy as np
import time
from IPython.display import display
import ipywidgets.widgets as widgets

print("Libraries loaded successfully")

In [ ]:
# =========================================
# CELL 2: Initialize camera and JetRacer (ROBUST VERSION)
# =========================================

import time
from jetcam.csi_camera import CSICamera
from jetracer.nvidia_racecar import NvidiaRacecar

# Initialize car
car = NvidiaRacecar()
car.throttle = 0
car.steering = 0

# Reset camera safely
try:
    camera.running = False
    time.sleep(1)
except:
    pass

camera = CSICamera(width=224, height=224, capture_fps=30)

# Force camera to stabilise
camera.running = True
time.sleep(2)

# Warm-up frames (THIS is what fixed your issue)
for _ in range(5):
    frame = camera.value
    time.sleep(0.2)

# Final check
frame = camera.value

print("Camera + Car ready")
print("Resolution:", camera.width, "x", camera.height)

if frame is None:
    print("Camera NOT working")
else:
    print("Camera working")
    print("Mean brightness:", round(frame.mean(), 2))

In [ ]:
# =========================================
# CELL 3: Image display widget
# =========================================
# Purpose:
# - Show camera feed in notebook
# - Required for debugging vision

image_widget = widgets.Image(format='jpeg', width=448, height=448)
display(image_widget)

def show_image(frame):
    _, jpeg = cv2.imencode('.jpg', frame)
    image_widget.value = jpeg.tobytes()

print("Display ready")

In [ ]:
# =========================================
# CELL 4: Camera test
# =========================================
# Purpose:
# - Verify camera is working
# - Check brightness levels

frame = camera.value

if frame is None:
    print("Camera NOT working")
else:
    print("Camera working")
    print("Shape:", frame.shape)
    print("Brightness (mean):", round(frame.mean(), 2))

    show_image(frame)

In [ ]:
# =========================================
# CELL 5: Adaptive white tape mask
# =========================================
# Purpose:
# - Use strict white detection first
# - If too little tape is found, use a slightly relaxed fallback
# - Avoid over-detecting carpet while still handling angled views

def get_white_mask(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Primary strict mask: clean when tape is clearly visible
    lower_strict = np.array([0, 0, 180])
    upper_strict = np.array([180, 60, 255])
    strict_mask = cv2.inRange(hsv, lower_strict, upper_strict)

    # Fallback relaxed mask: helps when tape is dim/angled
    lower_relaxed = np.array([0, 0, 165])
    upper_relaxed = np.array([180, 85, 255])
    relaxed_mask = cv2.inRange(hsv, lower_relaxed, upper_relaxed)

    # Measure strict mask amount
    strict_ratio = np.count_nonzero(strict_mask) / strict_mask.size

    # Use strict normally, fallback only if strict detection is too sparse
    if strict_ratio < 0.015:
        mask = relaxed_mask
    else:
        mask = strict_mask

    # Clean and reconnect tape segments
    kernel = np.ones((3, 3), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.dilate(mask, kernel, iterations=1)

    return mask

In [ ]:
# =========================================
# CELL 6: Adaptive mask diagnostic
# =========================================

frame = camera.value
mask = get_white_mask(frame)

white_pixels = np.count_nonzero(mask)
white_ratio = white_pixels / mask.size

print("White pixels:", white_pixels)
print("White ratio:", round(white_ratio, 4))

show_image(mask)

In [ ]:
# =========================================
# CELL 7: Robust entrance detector with quality gating
# =========================================

def detect_parking_bay(frame):
    height, width, _ = frame.shape
    image_center_x = width // 2

    mask = get_white_mask(frame)
    debug = frame.copy()

    roi_start = int(height * 0.25)
    roi_end = int(height * 0.90)
    roi = mask[roi_start:roi_end, :]

    candidates = []

    # -------- CONTOURS --------
    contours, _ = cv2.findContours(roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 15:
            continue

        x, y, w, h = cv2.boundingRect(cnt)
        if h < 8:
            continue

        pts = cnt.reshape(-1, 2)
        pts[:, 1] += roi_start

        max_y = np.max(pts[:, 1])
        bottom_pts = pts[pts[:, 1] >= max_y - 6]

        if len(bottom_pts) == 0:
            continue

        bx = int(np.median(bottom_pts[:, 0]))
        by = int(max_y)

        candidates.append({
            "x": bx,
            "y": by,
            "strength": area + h * 2,
            "source": "contour"
        })

    # -------- COLUMN FALLBACK --------
    column_sum = np.sum(roi > 0, axis=0)
    active = np.where(column_sum > 4)[0]

    if len(active) > 0:
        groups = []
        g = [active[0]]

        for x in active[1:]:
            if x - g[-1] <= 7:
                g.append(x)
            else:
                groups.append(g)
                g = [x]
        groups.append(g)

        for group in groups:
            cx = int(np.mean(group))
            strength = int(np.sum(column_sum[group]))

            if strength < 20:
                continue

            ys, _ = np.where(roi[:, group[0]:group[-1]+1] > 0)
            if len(ys) == 0:
                continue

            by = int(np.max(ys) + roi_start)

            candidates.append({
                "x": cx,
                "y": by,
                "strength": strength,
                "source": "column"
            })

    # -------- MERGE --------
    candidates = sorted(candidates, key=lambda c: c["strength"], reverse=True)
    merged = []

    for c in candidates:
        if not any(abs(c["x"] - m["x"]) < 12 for m in merged):
            merged.append(c)

    candidates = sorted(merged, key=lambda c: c["x"])

    # Draw candidates
    for c in candidates:
        color = (0,255,0) if c["source"]=="contour" else (0,255,255)
        cv2.circle(debug, (c["x"], c["y"]), 6, color, -1)

    cv2.line(debug, (image_center_x,0), (image_center_x,height), (0,0,255), 2)

    # -------- PAIRING --------
    best_pair = None
    best_score = 999999

    if len(candidates) >= 2:
        for i in range(len(candidates)):
            for j in range(i+1, len(candidates)):
                L = candidates[i]
                R = candidates[j]

                width_px = R["x"] - L["x"]
                if width_px < 40 or width_px > 180:
                    continue

                center_x = (L["x"] + R["x"]) // 2
                y_diff = R["y"] - L["y"]

                # CRITICAL: reject weak pairs
                if L["strength"] < 80 or R["strength"] < 80:
                    continue

                score = (
                    abs(center_x - image_center_x)*0.5 +
                    abs(width_px - 90)*0.3 +
                    abs(y_diff)*0.2
                )

                if score < best_score:
                    best_score = score
                    best_pair = (L, R, center_x, y_diff, width_px)

    # -------- OUTPUT --------
    if best_pair is not None:
        L, R, cx, y_diff, width_px = best_pair

        cv2.line(debug, (L["x"],L["y"]), (R["x"],R["y"]), (255,255,0), 2)
        cv2.circle(debug, (cx, (L["y"]+R["y"])//2), 8, (255,0,0), -1)

        return True, (cx, (L["y"]+R["y"])//2), y_diff, debug, {
            "mode": "FULL_BAY",
            "candidates": len(candidates)
        }

    elif len(candidates) >= 1:
        c = candidates[0]
        return True, (c["x"], c["y"]), None, debug, {
            "mode": "PARTIAL",
            "candidates": len(candidates)
        }

    return False, None, None, debug, {"mode":"NONE"}

In [ ]:
# =========================================
# CELL 8: Test entrance-point detection with candidate details
# =========================================

frame = camera.value

bay, entrance_center, angle_error, debug, diagnostics = detect_parking_bay(frame)

show_image(debug)

print("=== Entrance Detection Diagnostics ===")
for key, value in diagnostics.items():
    if key != "candidate_details":
        print(key, ":", value)

print("\nCandidate details:")
for idx, c in enumerate(diagnostics.get("candidate_details", [])):
    print(idx, c)

if entrance_center is not None:
    image_center_x = frame.shape[1] // 2
    x_error = entrance_center[0] - image_center_x

    print("\nImage center x:", image_center_x)
    print("Entrance center x:", entrance_center[0])
    print("Entrance center y:", entrance_center[1])
    print("X error:", x_error)
    print("Angle error y-diff:", angle_error)

In [ ]:
# =========================================
# CELL 9: Control constants and steering helper
# =========================================
# Purpose:
# - Store movement values in one place
# - Correct for steering drift
# - Provide simple centre-error steering

# Forward throttle values.
# Your JetRacer moves forward using negative throttle.
KICK_THROTTLE = -0.28       # short pulse to overcome carpet/floor resistance
DRIVE_THROTTLE = -0.22      # normal forward motion after kick
SLOW_THROTTLE = -0.18       # slower correction motion
REVERSE_THROTTLE = 0.22     # reverse motion

# Steering values.
# Adjust STEERING_OFFSET if the car naturally drifts left/right.
STEERING_OFFSET = -0.08

# Aggressive steering for correction manoeuvres.
# Use short pulses, not long continuous steering.
AGGRESSIVE_LEFT = 0.32
AGGRESSIVE_RIGHT = -0.32

# Centre tolerance.
# If bay centre is within this many pixels of image centre, it is considered aligned.
CENTER_TOLERANCE = 5

def calculate_center_error(bay_center, image_width):
    """
    Calculates horizontal error between bay centre and camera centre.
    
    Positive error:
        bay is to the right of camera centre.
        
    Negative error:
        bay is to the left of camera centre.
    """
    image_center = image_width // 2
    return bay_center - image_center

print("Control constants loaded")

In [ ]:
# =========================================
# CELL 10: Emergency stop
# =========================================

def stop():
    car.throttle = 0
    car.steering = 0
    print("STOPPED")

stop()

In [ ]:
# =========================================
# CELL 11: Stronger car-like movement primitives
# =========================================

def drive_forward(duration=0.35, steering=STEERING_OFFSET):
    car.steering = steering
    car.throttle = KICK_THROTTLE
    time.sleep(0.18)

    car.throttle = DRIVE_THROTTLE
    time.sleep(duration)

    stop()


def reverse_arc_left(duration=0.45):
    """
    Reverse while steering left.
    This changes heading more than reversing straight.
    """
    car.steering = 0.55
    car.throttle = REVERSE_THROTTLE
    time.sleep(duration)
    stop()


def reverse_arc_right(duration=0.45):
    """
    Reverse while steering right.
    """
    car.steering = -0.55
    car.throttle = REVERSE_THROTTLE
    time.sleep(duration)
    stop()


def forward_arc_left(duration=0.50):
    """
    Forward while steering left.
    Strong heading correction.
    """
    car.steering = 0.55
    car.throttle = KICK_THROTTLE
    time.sleep(0.18)

    car.throttle = DRIVE_THROTTLE
    time.sleep(duration)
    stop()


def forward_arc_right(duration=0.50):
    """
    Forward while steering right.
    Strong heading correction.
    """
    car.steering = -0.55
    car.throttle = KICK_THROTTLE
    time.sleep(0.18)

    car.throttle = DRIVE_THROTTLE
    time.sleep(duration)
    stop()


def correction_pulse(error):
    """
    Car-like correction:
    reverse opposite arc, then forward toward the bay.
    This changes orientation much more aggressively than the old pulse.
    """

    print("Correction pulse for error:", error)

    if error > CENTER_TOLERANCE:
        print("Bay is right: reverse left, forward right")
        reverse_arc_left(duration=0.45)
        forward_arc_right(duration=0.55)

    elif error < -CENTER_TOLERANCE:
        print("Bay is left: reverse right, forward left")
        reverse_arc_right(duration=0.45)
        forward_arc_left(duration=0.55)

    else:
        print("Within tolerance: no correction")

In [ ]:
# =========================================
# CELL 12: STRICT scan (FULL_BAY only)
# =========================================

def scan_for_bay(num_frames=8, delay=0.08):
    full_centers_x = []
    full_centers_y = []
    full_angles = []
    full_widths = []

    partial_seen = 0
    debug_frame = None

    for i in range(num_frames):
        frame = camera.value
        if frame is None:
            continue

        bay, center, angle, debug, diagnostics = detect_parking_bay(frame)
        debug_frame = debug

        mode = diagnostics.get("mode", "NONE")

        if mode == "FULL_BAY" and center is not None:
            full_centers_x.append(center[0])
            full_centers_y.append(center[1])

            if angle is not None:
                full_angles.append(angle)

            # width may not exist anymore, safe
            width = diagnostics.get("entrance_width", None)
            if width is not None:
                full_widths.append(width)

        elif mode == "PARTIAL":
            partial_seen += 1

        time.sleep(delay)

    # ---------- DECISION ----------
    if len(full_centers_x) > 0:
        avg_x = int(np.mean(full_centers_x))
        avg_y = int(np.mean(full_centers_y))

        avg_angle = int(np.mean(full_angles)) if len(full_angles) > 0 else None
        avg_width = int(np.mean(full_widths)) if len(full_widths) > 0 else None

        confidence = len(full_centers_x) / num_frames

        return True, (avg_x, avg_y), avg_angle, avg_width, confidence, debug_frame

    # fallback → only partial seen
    if partial_seen > 0:
        return True, None, None, None, partial_seen / num_frames, debug_frame

    return False, None, None, None, 0.0, debug_frame


print("STRICT scan (FULL_BAY only) ready")

In [ ]:
# =========================================
# CELL 12.5: Static entrance scan test
# =========================================

bay_found, entrance_center, angle_error, entrance_width, confidence, debug = scan_for_bay(num_frames=8)

if debug is not None:
    show_image(debug)

print("Bay found:", bay_found)
print("Entrance center:", entrance_center)
print("Angle error:", angle_error)
print("Entrance width:", entrance_width)
print("Confidence:", round(confidence, 2))

if entrance_center is not None:
    image_center_x = camera.width // 2
    x_error = entrance_center[0] - image_center_x

    print("Image center x:", image_center_x)
    print("X error:", x_error)
    print("Entrance Y:", entrance_center[1])

In [ ]:
# =========================================
# CELL 13 – STRICT STATE MACHINE CONTROLLER v2
# =========================================
# Key fix:
# - PARTIAL mode no longer turns away incorrectly.
# - If the visible edge is on the RIGHT, rotate RIGHT to open the bay view.
# - If the visible edge is on the LEFT, rotate LEFT.
# - Forward throttle uses NEGATIVE values.
# - Reverse throttle uses POSITIVE values.

print("Cell 13 started")

# -------------------------------
# Local movement functions
# -------------------------------

def stop_local():
    car.throttle = 0
    car.steering = 0
    time.sleep(0.1)
    print("STOPPED")


def forward_local(duration=0.30, steering=STEERING_OFFSET):
    """
    Forward = negative throttle on your JetRacer.
    """
    car.steering = steering

    # kick to overcome floor resistance
    car.throttle = KICK_THROTTLE
    time.sleep(0.12)

    car.throttle = DRIVE_THROTTLE
    time.sleep(duration)

    stop_local()


def reverse_local(duration=0.30, steering=STEERING_OFFSET):
    """
    Reverse = positive throttle on your JetRacer.
    """
    car.steering = steering
    car.throttle = REVERSE_THROTTLE
    time.sleep(duration)

    stop_local()


def rotate_left_local(duration=0.35):
    """
    Strong forward-left arc.
    """
    car.steering = 0.60
    car.throttle = KICK_THROTTLE
    time.sleep(0.12)

    car.throttle = DRIVE_THROTTLE
    time.sleep(duration)

    stop_local()


def rotate_right_local(duration=0.35):
    """
    Strong forward-right arc.
    """
    car.steering = -0.60
    car.throttle = KICK_THROTTLE
    time.sleep(0.12)

    car.throttle = DRIVE_THROTTLE
    time.sleep(duration)

    stop_local()


# -------------------------------
# Controller parameters
# -------------------------------

IMAGE_CENTER = camera.width // 2

X_TOL = 5
ANGLE_TOL = 8
TARGET_Y = 150

MAX_STEPS = 50
step = 0

try:
    while step < MAX_STEPS:
        step += 1
        print("\n--- STEP", step, "---")

        stop_local()

        bay_found, entrance_center, angle_error, width, conf, debug = scan_for_bay(num_frames=6)

        if debug is not None:
            show_image(debug)

        print(
            "SCAN | found:", bay_found,
            "| center:", entrance_center,
            "| angle:", angle_error,
            "| width:", width,
            "| conf:", round(conf, 2)
        )

        # -------------------------------
        # CASE 1: NO BAY
        # -------------------------------
        if not bay_found:
            print("NO BAY → slow search rotate RIGHT")
            rotate_right_local(duration=0.30)
            continue

        # -------------------------------
        # CASE 2: PARTIAL BAY
        # -------------------------------
        if width is None:
            print("PARTIAL BAY → recovery mode. NO forward entry.")

            # reverse first to create room
            reverse_local(duration=0.30, steering=STEERING_OFFSET)

            if entrance_center is None:
                print("No usable edge centre → default rotate RIGHT")
                rotate_right_local(duration=0.35)
                continue

            edge_error = entrance_center[0] - IMAGE_CENTER
            print("PARTIAL EDGE ERROR:", edge_error)

            # IMPORTANT FIX:
            # Previous behaviour turned left when edge was right.
            # That caused it to hug the left tape and drift out.
            if edge_error > 0:
                print("Visible edge is RIGHT → rotate RIGHT")
                rotate_right_local(duration=0.45)
            else:
                print("Visible edge is LEFT → rotate LEFT")
                rotate_left_local(duration=0.45)

            continue

        # -------------------------------
        # CASE 3: FULL BAY
        # -------------------------------
        x_error = entrance_center[0] - IMAGE_CENTER

        if angle_error is None:
            angle_error = 0

        print("FULL BAY | X ERROR:", x_error, "| ANGLE ERROR:", angle_error)

        # Fix angle first
        if abs(angle_error) > ANGLE_TOL:
            print("ANGLE correction")

            reverse_local(duration=0.25, steering=STEERING_OFFSET)

            if angle_error < 0:
                print("Angle negative → rotate RIGHT")
                rotate_right_local(duration=0.35)
            else:
                print("Angle positive → rotate LEFT")
                rotate_left_local(duration=0.35)

            continue

        # Then fix x position
        if abs(x_error) > X_TOL:
            print("X correction")

            reverse_local(duration=0.20, steering=STEERING_OFFSET)

            if x_error > 0:
                print("Bay centre RIGHT → rotate RIGHT")
                rotate_right_local(duration=0.30)
            else:
                print("Bay centre LEFT → rotate LEFT")
                rotate_left_local(duration=0.30)

            continue

        # Approach only if FULL BAY exists and aligned
        if entrance_center[1] < TARGET_Y:
            print("FULL + aligned but far → cautious forward")
            forward_local(duration=0.25, steering=STEERING_OFFSET)
            continue

        print("SUCCESS: aligned close to entrance")
        stop_local()
        break

    stop_local()
    print("Controller finished")

except KeyboardInterrupt:
    stop_local()
    print("Stopped manually")